# DIP-Pipeline: Kontextvalidierung & Rohdaten-Ingestion

Themenfilter: `Rente`, `Altersabsicherung`

Benötigte Pakete: `requests`, `pyyaml` (Standardbibliothek: `sqlite3`, `hashlib`, `json`, `os`, `time`, `datetime`)

Vor dem Ausführen muss die Umgebungsvariable `DIP_API_KEY` gesetzt sein.

**Zwei API-Einschränkungen, die die Abfrage-Logik unten prägen:**
- `/vorgang` bietet keinen Volltext-/Stichwort-Parameter → Keyword-Filterung erfolgt client-seitig auf `titel`/`abstract`.
- `/drucksache` und `/plenarprotokoll` haben keinen `f.vorgang`-Filter → die zugehörigen Dokumente werden über `fundstelle` der Vorgangspositionen ermittelt und gebündelt per `f.id` nachgeladen.

## 0. Setup

In [ ]:
import hashlib
import json
import os
import sqlite3
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import requests
import yaml

DIP_BASE_URL = "https://search.dip.bundestag.de/api/v1"
DIP_OPENAPI_URL = "https://search.dip.bundestag.de/api/v1/openapi.yaml"
DIP_NUTZUNGSBEDINGUNGEN_URL = "https://dip.bundestag.de/documents/nutzungsbedingungen_dip.pdf"

API_KEY = os.environ["DIP_API_KEY"]

DB_PATH = Path("dip_rohdaten.sqlite")

SUCHBEGRIFFE = ["Rente", "Altersabsicherung", "Altersvorsorge", "Altersarmut", "Rentenversicherung", "Rentenreform", "Rentenanpassung"]
UPDATE_BUFFER_MINUTEN = 15
INITIAL_ZEITRAUM_TAGE = 30

**Datenbankschema:** je eine Rohdaten-Tabelle pro Ressourcentyp (`id` als Primärschlüssel, `aktualisiert` für den Update-Abgleich, `raw_json` als vollständiger API-Response – Bereinigung/Minimierung passiert erst später im dbt-Modelling). Dazu drei Betriebstabellen: `kontext_validierung_log` (Protokoll der Kontextchecks), `kontext_referenzwerte` (letzter bekannter Hash/Snapshot je Check) und `sync_state` (Zeitpunkt der letzten erfolgreichen Abfrage je Ressourcentyp).

In [ ]:
def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA journal_mode = WAL;")
    return conn


def init_db():
    conn = get_connection()
    conn.executescript(
        """
        CREATE TABLE IF NOT EXISTS vorgang (
            id TEXT PRIMARY KEY,
            aktualisiert TEXT NOT NULL,
            titel TEXT,
            datum TEXT,
            raw_json TEXT NOT NULL,
            geladen_am TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS vorgangsposition (
            id TEXT PRIMARY KEY,
            vorgang_id TEXT NOT NULL,
            aktualisiert TEXT NOT NULL,
            datum TEXT,
            raw_json TEXT NOT NULL,
            geladen_am TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS drucksache (
            id TEXT PRIMARY KEY,
            aktualisiert TEXT NOT NULL,
            dokumentnummer TEXT,
            datum TEXT,
            raw_json TEXT NOT NULL,
            geladen_am TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS plenarprotokoll (
            id TEXT PRIMARY KEY,
            aktualisiert TEXT NOT NULL,
            dokumentnummer TEXT,
            datum TEXT,
            raw_json TEXT NOT NULL,
            geladen_am TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS kontext_validierung_log (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            zeitpunkt TEXT NOT NULL,
            pruefung TEXT NOT NULL,
            status TEXT NOT NULL,
            details TEXT
        );

        CREATE TABLE IF NOT EXISTS kontext_referenzwerte (
            schluessel TEXT PRIMARY KEY,
            wert TEXT NOT NULL,
            gesetzt_am TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS sync_state (
            ressourcentyp TEXT PRIMARY KEY,
            letzte_abfrage TEXT NOT NULL
        );
        """
    )
    conn.commit()
    conn.close()


init_db()

## 1. Kontextvalidierung

Drei unabhängige, deterministische Vor-Checks, die vor jedem Pipeline-Lauf ausgeführt werden. Jeder Check schreibt sein Ergebnis in `kontext_validierung_log` und aktualisiert seinen Referenzwert in `kontext_referenzwerte`. Status `geaendert` oder `fehler` soll den eigentlichen Datenabruf blockieren (siehe Abschnitt 3).

**OpenAPI-Schema-Check** – lädt die aktuelle OpenAPI-Spezifikation, vergleicht Endpunktliste und Schema-Hashes gegen den letzten bekannten Snapshot und protokolliert konkrete Änderungen (neue/entfernte Endpunkte, neue/geänderte Schemas).

In [ ]:
def openapi_schema_check(conn):
    response = requests.get(DIP_OPENAPI_URL, timeout=30)
    response.raise_for_status()
    schema = yaml.safe_load(response.text)

    endpunkte = sorted(schema.get("paths", {}).keys())
    schema_hashes = {
        name: hashlib.sha256(json.dumps(definition, sort_keys=True).encode()).hexdigest()
        for name, definition in schema.get("components", {}).get("schemas", {}).items()
    }

    referenz_zeile = conn.execute(
        "SELECT wert FROM kontext_referenzwerte WHERE schluessel = 'openapi_snapshot'"
    ).fetchone()

    aenderungen = []
    if referenz_zeile is not None:
        referenz = json.loads(referenz_zeile[0])
        neue_endpunkte = sorted(set(endpunkte) - set(referenz["endpunkte"]))
        entfernte_endpunkte = sorted(set(referenz["endpunkte"]) - set(endpunkte))
        neue_schemas = sorted(set(schema_hashes) - set(referenz["schema_hashes"]))
        geaenderte_schemas = sorted(
            name
            for name, wert in schema_hashes.items()
            if name in referenz["schema_hashes"] and referenz["schema_hashes"][name] != wert
        )

        if neue_endpunkte:
            aenderungen.append(f"neue Endpunkte: {neue_endpunkte}")
        if entfernte_endpunkte:
            aenderungen.append(f"entfernte Endpunkte: {entfernte_endpunkte}")
        if neue_schemas:
            aenderungen.append(f"neue Schemas: {neue_schemas}")
        if geaenderte_schemas:
            aenderungen.append(f"geänderte Schemas: {geaenderte_schemas}")

    status = "geaendert" if aenderungen else "ok"
    details = "; ".join(aenderungen) if aenderungen else "keine strukturelle Änderung festgestellt"

    conn.execute(
        "INSERT INTO kontext_validierung_log (zeitpunkt, pruefung, status, details) VALUES (?, 'openapi_schema', ?, ?)",
        (datetime.now(timezone.utc).isoformat(), status, details),
    )
    conn.execute(
        """
        INSERT OR REPLACE INTO kontext_referenzwerte (schluessel, wert, gesetzt_am)
        VALUES ('openapi_snapshot', ?, ?)
        """,
        (
            json.dumps({"endpunkte": endpunkte, "schema_hashes": schema_hashes}),
            datetime.now(timezone.utc).isoformat(),
        ),
    )
    conn.commit()
    return status

**API-Key-Check** – führt eine minimale, echte Testanfrage aus (statt Website-Scraping) und wertet den HTTP-Statuscode aus: `200` = gültig, `401` = Key wurde offenbar geändert/ist ungültig.

In [ ]:
def api_key_check(conn):
    try:
        response = requests.get(
            f"{DIP_BASE_URL}/vorgang",
            params={"apikey": API_KEY, "f.id": 1},
            timeout=15,
        )
        if response.status_code == 200:
            status, details = "ok", "API-Key gültig"
        elif response.status_code == 401:
            status, details = "geaendert", "API-Key wird abgelehnt (401) - vermutlich geändert/abgelaufen"
        else:
            status, details = "fehler", f"unerwarteter Statuscode {response.status_code} beim Key-Check"
    except requests.RequestException as fehler:
        status, details = "fehler", f"Netzwerkfehler beim Key-Check: {fehler}"

    conn.execute(
        "INSERT INTO kontext_validierung_log (zeitpunkt, pruefung, status, details) VALUES (?, 'api_key', ?, ?)",
        (datetime.now(timezone.utc).isoformat(), status, details),
    )
    conn.commit()
    return status

**Nutzungsbedingungen-Check** – lädt das PDF der Nutzungsbedingungen, hasht den Inhalt und vergleicht gegen den letzten bekannten Hash.

In [ ]:
def nutzungsbedingungen_check(conn):
    response = requests.get(DIP_NUTZUNGSBEDINGUNGEN_URL, timeout=30)
    response.raise_for_status()
    aktueller_hash = hashlib.sha256(response.content).hexdigest()

    referenz = conn.execute(
        "SELECT wert FROM kontext_referenzwerte WHERE schluessel = 'nutzungsbedingungen_hash'"
    ).fetchone()

    if referenz is None:
        status, details = "ok", "Erststart: Referenzwert gesetzt, kein Vergleich möglich"
    elif referenz[0] == aktueller_hash:
        status, details = "ok", "keine Änderung der Nutzungsbedingungen festgestellt"
    else:
        status, details = "geaendert", "Nutzungsbedingungen-PDF hat sich verändert - manuelle Prüfung nötig"

    conn.execute(
        """
        INSERT INTO kontext_validierung_log (zeitpunkt, pruefung, status, details)
        VALUES (?, 'nutzungsbedingungen', ?, ?)
        """,
        (datetime.now(timezone.utc).isoformat(), status, details),
    )
    conn.execute(
        """
        INSERT OR REPLACE INTO kontext_referenzwerte (schluessel, wert, gesetzt_am)
        VALUES ('nutzungsbedingungen_hash', ?, ?)
        """,
        (aktueller_hash, datetime.now(timezone.utc).isoformat()),
    )
    conn.commit()
    return status

**Sammelfunktion** – führt alle drei Checks aus und bricht mit einer Exception ab, falls mindestens einer `geaendert` oder `fehler` meldet. Details stehen in `kontext_validierung_log`.

In [ ]:
def kontext_validierung_durchfuehren():
    conn = get_connection()
    ergebnisse = {
        "openapi_schema": openapi_schema_check(conn),
        "api_key": api_key_check(conn),
        "nutzungsbedingungen": nutzungsbedingungen_check(conn),
    }
    conn.close()

    kritisch = [name for name, status in ergebnisse.items() if status in {"geaendert", "fehler"}]
    if kritisch:
        raise RuntimeError(
            f"Kontextvalidierung fehlgeschlagen für: {kritisch} - Details siehe kontext_validierung_log"
        )
    return ergebnisse

## 2. API-Request und Rohdaten-Speicherung

**Generische Abfragefunktion** – kapselt Cursor-Pagination (Folgeanfragen bis der Cursor sich nicht mehr ändert) sowie Retry mit exponentiellem Backoff bei `429`/`5xx`. `params` kann Listenwerte enthalten (z.B. `f.id`), `requests` wiederholt den Parameter dann automatisch.

In [ ]:
def _request_mit_retry(url, params, max_versuche=5):
    wartezeit = 1.0
    for versuch in range(1, max_versuche + 1):
        response = requests.get(url, params=params, timeout=30)
        if response.status_code == 200:
            return response
        if response.status_code in (429, 500, 502, 503, 504) and versuch < max_versuche:
            time.sleep(wartezeit)
            wartezeit *= 2
            continue
        response.raise_for_status()
    raise RuntimeError(f"Anfrage an {url} nach {max_versuche} Versuchen fehlgeschlagen")


def dip_request(ressourcentyp, params):
    basis_params = {"apikey": API_KEY, **params}
    alle_dokumente = []
    cursor = None

    while True:
        anfrage_params = dict(basis_params)
        if cursor is not None:
            anfrage_params["cursor"] = cursor

        response = _request_mit_retry(f"{DIP_BASE_URL}/{ressourcentyp}", anfrage_params)
        daten = response.json()
        alle_dokumente.extend(daten.get("documents", []))

        neuer_cursor = daten.get("cursor")
        if neuer_cursor is None or neuer_cursor == cursor:
            break
        cursor = neuer_cursor
        time.sleep(0.2)

    return alle_dokumente

**Sync-State-Hilfsfunktionen** – lesen/schreiben den Zeitpunkt der letzten erfolgreichen Abfrage je Ressourcentyp; Basis für das `f.aktualisiert.start`-Fenster inkl. 15-Minuten-Überlappungspuffer.

In [ ]:
def letzte_abfrage_holen(conn, ressourcentyp, standard_zeitpunkt):
    zeile = conn.execute(
        "SELECT letzte_abfrage FROM sync_state WHERE ressourcentyp = ?", (ressourcentyp,)
    ).fetchone()
    return zeile[0] if zeile else standard_zeitpunkt


def letzte_abfrage_setzen(conn, ressourcentyp, zeitpunkt):
    conn.execute(
        "INSERT OR REPLACE INTO sync_state (ressourcentyp, letzte_abfrage) VALUES (?, ?)",
        (ressourcentyp, zeitpunkt),
    )
    conn.commit()

**Vorgänge abfragen** – zieht Vorgänge über `f.aktualisiert.start` (initial: letzte 30 Tage, danach: seit letzter Abfrage minus Puffer), filtert client-seitig auf die Suchbegriffe in `titel`/`abstract` und speichert Treffer per Upsert (`INSERT OR REPLACE`) in SQLite.

In [ ]:
def enthaelt_suchbegriff(vorgang):
    text = " ".join(filter(None, [vorgang.get("titel"), vorgang.get("abstract")])).lower()
    return any(begriff.lower() in text for begriff in SUCHBEGRIFFE)


def vorgaenge_abfragen_und_speichern(conn):
    lauf_start = datetime.now(timezone.utc)
    standard_start = (lauf_start - timedelta(days=INITIAL_ZEITRAUM_TAGE)).isoformat()
    letzte_abfrage = letzte_abfrage_holen(conn, "vorgang", standard_start)
    abfrage_start = (
        datetime.fromisoformat(letzte_abfrage) - timedelta(minutes=UPDATE_BUFFER_MINUTEN)
    ).isoformat()

    rohdaten = dip_request("vorgang", {"f.aktualisiert.start": abfrage_start})
    relevante_vorgaenge = [vorgang for vorgang in rohdaten if enthaelt_suchbegriff(vorgang)]

    for vorgang in relevante_vorgaenge:
        conn.execute(
            """
            INSERT OR REPLACE INTO vorgang (id, aktualisiert, titel, datum, raw_json, geladen_am)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (
                vorgang["id"],
                vorgang["aktualisiert"],
                vorgang.get("titel"),
                vorgang.get("datum"),
                json.dumps(vorgang, ensure_ascii=False),
                datetime.now(timezone.utc).isoformat(),
            ),
        )
    conn.commit()

    letzte_abfrage_setzen(conn, "vorgang", lauf_start.isoformat())
    return relevante_vorgaenge

**Kinddaten je Vorgang** – holt die Vorgangspositionen per `f.vorgang=<id>`, sammelt daraus die `fundstelle`-IDs (getrennt nach `Drucksache`/`Plenarprotokoll`) und lädt die zugehörigen Dokumente gebündelt per wiederholtem `f.id`-Parameter nach. Läuft für jeden in diesem Durchlauf gefundenen bzw. aktualisierten Vorgang.

In [ ]:
def _dokumente_per_id_batch(ressourcentyp, ids, batch_groesse=50):
    ids = sorted(ids)
    dokumente = []
    for start in range(0, len(ids), batch_groesse):
        batch = ids[start : start + batch_groesse]
        dokumente.extend(dip_request(ressourcentyp, {"f.id": batch}))
    return dokumente


def kinddaten_abfragen_und_speichern(conn, vorgang_id):
    vorgangspositionen = dip_request("vorgangsposition", {"f.vorgang": vorgang_id})
    fundstellen_ids = {"Drucksache": set(), "Plenarprotokoll": set()}

    for vp in vorgangspositionen:
        conn.execute(
            """
            INSERT OR REPLACE INTO vorgangsposition (id, vorgang_id, aktualisiert, datum, raw_json, geladen_am)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (
                vp["id"],
                vp["vorgang_id"],
                vp["aktualisiert"],
                vp.get("datum"),
                json.dumps(vp, ensure_ascii=False),
                datetime.now(timezone.utc).isoformat(),
            ),
        )
        fundstelle = vp.get("fundstelle")
        if fundstelle:
            fundstellen_ids[fundstelle["dokumentart"]].add(fundstelle["id"])

    for drucksache in _dokumente_per_id_batch("drucksache", fundstellen_ids["Drucksache"]):
        conn.execute(
            """
            INSERT OR REPLACE INTO drucksache (id, aktualisiert, dokumentnummer, datum, raw_json, geladen_am)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (
                drucksache["id"],
                drucksache["aktualisiert"],
                drucksache.get("dokumentnummer"),
                drucksache.get("datum"),
                json.dumps(drucksache, ensure_ascii=False),
                datetime.now(timezone.utc).isoformat(),
            ),
        )

    for protokoll in _dokumente_per_id_batch("plenarprotokoll", fundstellen_ids["Plenarprotokoll"]):
        conn.execute(
            """
            INSERT OR REPLACE INTO plenarprotokoll (id, aktualisiert, dokumentnummer, datum, raw_json, geladen_am)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (
                protokoll["id"],
                protokoll["aktualisiert"],
                protokoll.get("dokumentnummer"),
                protokoll.get("datum"),
                json.dumps(protokoll, ensure_ascii=False),
                datetime.now(timezone.utc).isoformat(),
            ),
        )

    conn.commit()

**Orchestrierung** – verbindet beide Schritte: erst die (inkrementell gefilterten) Vorgänge laden, dann für jeden davon die Kinddaten nachziehen.

In [ ]:
def vorgaenge_und_kinddaten_synchronisieren():
    conn = get_connection()
    gefundene_vorgaenge = vorgaenge_abfragen_und_speichern(conn)

    for vorgang in gefundene_vorgaenge:
        kinddaten_abfragen_und_speichern(conn, vorgang["id"])

    conn.close()
    return len(gefundene_vorgaenge)

## 3. Pipeline-Einstiegspunkt (für den Scheduler)

Kontextvalidierung läuft vor jedem Datenabruf und blockiert diesen bei kritischem Ergebnis. Diese Funktion ist der Einstiegspunkt, den ein Scheduler (z.B. APScheduler oder cron + papermill) periodisch aufruft.

In [ ]:
def pipeline_run():
    kontext_validierung_durchfuehren()
    anzahl_vorgaenge = vorgaenge_und_kinddaten_synchronisieren()
    return anzahl_vorgaenge


pipeline_run()